1번 BF파일과 DoS 데이터셋 통합! 
하나의 머신러닝으로 구현 후 공격 분류하기 위함!!!

input데이터
1. BF_new.csv : (CIC-BruteForce데이터셋 + 가상 환경에서 BruteForce 공격 패킷 수집 데이터셋)에 전처리 적용. 전처리 코드= ML팀/전처리/합병/merge.ipynb)
2. DoS_new.csv : (CIC-DoS데이터셋 + 가상 환경에서 DoS Slowloris 공격 패킷 수집 데이터셋)에 전처리 적용. 전처리 코드= ML팀/전처리/합병/merge.ipynb)

output데이터
1. finaldataset.csv

In [3]:
import pandas as pd

bf = pd.read_csv("BF_new.csv")
bf.columns = bf.columns.str.strip()

dos = pd.read_csv("DoS_new.csv")
dos.columns = dos.columns.str.strip()

# 컬럼 구성 다른지 체크
print("BF에만 있는 컬럼:", set(bf.columns) - set(dos.columns))
print("DoS에만 있는 컬럼:", set(dos.columns) - set(bf.columns))

combined = pd.concat([bf, dos], ignore_index=True)

# BENIGN이 두 파일 모두에 들어있어서 완전히 똑같은 행이 겹칠 수도 있으니 중복 제거
before = len(combined)
combined = combined.drop_duplicates()
print(f"중복 제거: {before - len(combined)}행")

combined.to_csv("final_dataset.csv", index=False)
print(combined["Label"].value_counts())

BF에만 있는 컬럼: set()
DoS에만 있는 컬럼: set()
중복 제거: 41574행
Label
BENIGN         566579
DoS              9433
Brute Force      7653
Name: count, dtype: int64


2번 위에서 통합한 csv 파일 ML(catboost) 학습 후 pkl 생성

input데이터
1. final_dataset.csv

output데이터
1. multicast_cat.pkl

In [4]:
#통합 데이터 ML 학습
import re
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report

df = pd.read_csv(r"C:\python_proj\모듈 프로젝트2\ML\완성\final_dataset.csv")
df.columns = df.columns.str.strip()
X = df.drop(columns=["Label"])
y = df["Label"]

# 컬럼 이름 정규화 => 특수문자를 공백 / _ 로 변환
X.columns = [re.sub(r"[^\w\s]", "", col).strip().replace(" ", "_") for col in X.columns]

# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 캣부스트 모델 생성
model = CatBoostClassifier(auto_class_weights="Balanced", random_state=42, verbose=100)

# 모델 학습
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 클래스별 성능 (BENIGN/Brute Force/DoS 각각 precision/recall/f1 확인용)
print(classification_report(y_test, y_pred))

# 전체 요약 지표 (클래스 불균형 반영)
f1_weighted = f1_score(y_test, y_pred, average="weighted")
print(f"Catboost Weighted F1: {f1_weighted:.4f}")

# 학습된 모델을 pkl 파일로 저장
with open("multiclass_cat.pkl", "wb") as f:
    pickle.dump(model, f)

print("multiclass_cat.pkl 저장 완료")

Learning rate set to 0.108257
0:	learn: 0.9080115	total: 253ms	remaining: 4m 12s
100:	learn: 0.0022247	total: 14.1s	remaining: 2m 5s
200:	learn: 0.0004598	total: 27.9s	remaining: 1m 50s
300:	learn: 0.0001944	total: 40.6s	remaining: 1m 34s
400:	learn: 0.0001202	total: 52.9s	remaining: 1m 19s
500:	learn: 0.0000812	total: 1m 5s	remaining: 1m 5s
600:	learn: 0.0000612	total: 1m 18s	remaining: 52.1s
700:	learn: 0.0000508	total: 1m 30s	remaining: 38.7s
800:	learn: 0.0000422	total: 1m 43s	remaining: 25.7s
900:	learn: 0.0000359	total: 1m 55s	remaining: 12.7s
999:	learn: 0.0000307	total: 2m 9s	remaining: 0us
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    113316
 Brute Force       1.00      1.00      1.00      1531
         DoS       1.00      1.00      1.00      1886

    accuracy                           1.00    116733
   macro avg       1.00      1.00      1.00    116733
weighted avg       1.00      1.00      1.00    116733

Catboost Weig

## 3번과 4번은 같은 코드이며 input 테스트 데이터만 BF 환경에서 수집됐는지 Slowloris 환경에서 수집되었는지만 다릅니다.

3번 ML모델에 테스트용 BruteForce어택 csv 분석(학습용과 별개로 vmware에서 새로 수집한 BF 공격 패킷 데이터)

input데이터
1. multicast_cat.pkl(전체 데이터 학습한 ML 모델)
2. bftest.csv(테스트용 데이터)

output데이터
1. bftest_prediction.csv

In [2]:
#성능 분석-BF 테스트 데이터 투입
import re
import pickle
import pandas as pd

# 모델 로드
with open(r"C:\python_proj\모듈 프로젝트2\ML\완성\multiclass_cat.pkl", "rb") as f:
    model = pickle.load(f)

# 예측할 테스트 데이터 로드 (매핑 스크립트 거친 캡처 csv)
test_path = r"C:\python_proj\모듈 프로젝트2\ML\완성\bftest.csv"
output_path = r"C:\python_proj\모듈 프로젝트2\ML\완성\bftest_prediction.csv"

df = pd.read_csv(test_path)
df.columns = df.columns.str.strip()

# 원본 데이터 보존 (라벨/기타 컬럼 포함)
df_original = df.copy()

# 학습 때와 동일하게 컬럼명 정규화
df.columns = [re.sub(r"[^\w\s]", "", col).strip().replace(" ", "_") for col in df.columns]

# 모델이 학습한 피처 목록 기준으로 컬럼 맞추기
model_features = model.feature_names_
missing = set(model_features) - set(df.columns)
if missing:
    print("모델에 필요한데 테스트 데이터에 없는 컬럼:", missing)

X_test = df[model_features]

# 예측
y_pred = model.predict(X_test).ravel()
y_proba = model.predict_proba(X_test)

df_original["Predicted_Label"] = y_pred
for i, cls in enumerate(model.classes_):
    df_original[f"Prob_{cls}"] = y_proba[:, i]
df_original.to_csv(output_path, index=False)

print(df_original["Predicted_Label"].value_counts())
print(f"결과 저장 완료: {output_path}")

Predicted_Label
Brute Force    2023
DoS             118
Name: count, dtype: int64
결과 저장 완료: C:\python_proj\모듈 프로젝트2\ML\완성\bftest_prediction.csv


4번 ML모델에 테스트용 DoS Slowloris어택 csv 분석(학습용과 별개로 vmware에서 새로 수집한 SlowLoris 공격 패킷 데이터)

input데이터
1. multicast_cat.pkl(전체 데이터 학습한 ML 모델)
2. slowtest.csv(테스트용 데이터)

output데이터
1. slowtest_prediction.csv

In [1]:
#성능 분석-slow 테스트 데이터 투입
import re
import pickle
import pandas as pd

# 모델 로드
with open(r"C:\python_proj\모듈 프로젝트2\ML\완성\multiclass_cat.pkl", "rb") as f:
    model = pickle.load(f)

# 예측할 테스트 데이터 로드 (매핑 스크립트 거친 캡처 csv)
test_path = r"C:\python_proj\모듈 프로젝트2\ML\완성\slowtest.csv"
output_path = r"C:\python_proj\모듈 프로젝트2\ML\완성\slowtest_prediction.csv"

df = pd.read_csv(test_path)
df.columns = df.columns.str.strip()

# 원본 데이터 보존 (라벨/기타 컬럼 포함)
df_original = df.copy()

# 학습 때와 동일하게 컬럼명 정규화
df.columns = [re.sub(r"[^\w\s]", "", col).strip().replace(" ", "_") for col in df.columns]

# 모델이 학습한 피처 목록 기준으로 컬럼 맞추기
model_features = model.feature_names_
missing = set(model_features) - set(df.columns)
if missing:
    print("모델에 필요한데 테스트 데이터에 없는 컬럼:", missing)

X_test = df[model_features]

# 예측
y_pred = model.predict(X_test).ravel()
y_proba = model.predict_proba(X_test)

df_original["Predicted_Label"] = y_pred
for i, cls in enumerate(model.classes_):
    df_original[f"Prob_{cls}"] = y_proba[:, i]
df_original.to_csv(output_path, index=False)

print(df_original["Predicted_Label"].value_counts())
print(f"결과 저장 완료: {output_path}")

Predicted_Label
DoS       171
BENIGN      2
Name: count, dtype: int64
결과 저장 완료: C:\python_proj\모듈 프로젝트2\ML\완성\slowtest_prediction.csv
